# Lab 07: Improving HOG-Based Medical Image Classification Using a Genetic Algorithm

**Research question:** Can Genetic Algorithm (GA) based feature selection improve the classification performance of HOG features while reducing feature dimensionality on medical images?

**Pipelines compared (identical data split, identical HOG, identical SVM family):**
* **Baseline:** `Image → Preprocess (128×128) → HOG (8100-D) → StandardScaler → RBF-SVM`
* **Proposed:** `Image → Preprocess (128×128) → HOG (8100-D) → GA feature mask → StandardScaler → RBF-SVM`

**Contents**
1. Setup & configuration
2. Dataset (PneumoniaMNIST) and the fixed train / validation / test split
3. Preprocessing
4. HOG feature extraction (+ visualisation)
5. Baseline HOG + SVM (hyper-parameters tuned on the validation set)
6. Genetic Algorithm for HOG feature selection (initialisation, fitness, selection, crossover, mutation)
7. GA experiments with three GA settings + model selection on the validation set
8. GA-HOG + SVM (re-tuned on validation, trained on the selected features)
9. Final test-set evaluation (run **once**, after all model selection) – Tables 1 & 2
10. Plots: GA fitness, feature counts, confusion matrices, selected-feature map, computational time
11. Report: discussion, strengths, limitations, computational cost, conclusion

**Running on Google Colab (recommended):** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all` (≈ 5–10 min).
The SVM kernel matrices for the hundreds of GA fitness evaluations are computed on the GPU with PyTorch, and the libsvm fits run in parallel CPU threads. Without a GPU the notebook still runs (same code, PyTorch on CPU), just slower.

## 1. Setup & configuration

In [ ]:
%pip install -q kagglehub scikit-image

In [ ]:
import os, time, hashlib, warnings, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np, pandas as pd, cv2, torch
from IPython.display import Markdown
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from skimage.feature import hog
from skimage import exposure
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score,
                             cohen_kappa_score, matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)
warnings.filterwarnings("ignore")

SEED = 42                       # fixed seed for every random step (GA, random-subset control)
np.random.seed(SEED)
IMG_SIZE = 128                  # Table 1: 128 x 128
HOG_PARAMS = dict(orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), block_norm="L2-Hys")
CLASS_NAMES = ["Normal", "Pneumonia"]

# SVM hyper-parameter grid (tuned on the validation set for BOTH pipelines)
#   gamma = g / n_features  ->  g = 1 is sklearn's gamma="scale" on standardised features,
#   so the kernel width adapts automatically when the GA removes features.
C_GRID = [0.1, 1, 10, 100]
G_GRID = [1 / 64, 1 / 32, 1 / 16, 1 / 8, 1 / 4, 1 / 2, 1, 2]

OUT = Path("results"); OUT.mkdir(exist_ok=True)
pd.set_option("display.precision", 4)

# Compute device: kernel matrices on the GPU (T4 on Colab); libsvm fits in parallel CPU threads
# (libsvm releases the GIL, so threads give a real speed-up).
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = False      # full FP32 precision for the distance matrices
N_WORKERS = os.cpu_count() or 2
POOL = ThreadPoolExecutor(N_WORKERS)
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else
                             "  -> no GPU found; in Colab use Runtime > Change runtime type > T4 GPU"))
print(f"CPU threads for libsvm: {N_WORKERS} | torch {torch.__version__}")

## 2. Dataset: PneumoniaMNIST

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rijulshr/pneumoniamnist")

print("Path to dataset files:", path)
data = np.load(Path(path) / "pneumoniamnist.npz")
print({k: data[k].shape for k in data.files})

The `.npz` file already contains a **fixed, patient-independent train / validation / test split** (the MedMNIST split). We use it unchanged for *every* approach, so the baseline and the GA model see exactly the same images:
* **train** – fit the scaler and the SVMs
* **validation** – SVM hyper-parameter tuning, GA fitness, choice of the GA setting (all model selection)
* **test** – touched **once**, in Section 9, after everything has been selected

In [ ]:
SPLITS = ("train", "val", "test")
X_img = {s: data[f"{s}_images"] for s in SPLITS}
y = {s: data[f"{s}_labels"].ravel().astype(int) for s in SPLITS}

split_df = pd.DataFrame([{"Split": s, "Images": len(y[s]), "Normal (0)": int((y[s] == 0).sum()),
                          "Pneumonia (1)": int((y[s] == 1).sum()), "Pneumonia %": round(100 * y[s].mean(), 1)}
                         for s in SPLITS]).set_index("Split")
display(split_df)
split_df.to_csv(OUT / "dataset_split.csv")

# leakage check: exact-duplicate images shared between splits
h = {s: {hashlib.md5(im.tobytes()).hexdigest() for im in X_img[s]} for s in SPLITS}
DUPS = {"train∩val": len(h["train"] & h["val"]), "train∩test": len(h["train"] & h["test"]), "val∩test": len(h["val"] & h["test"])}
print("Exact-duplicate images shared between splits:", DUPS)
print("Image dtype / range:", X_img["train"].dtype, X_img["train"].min(), "-", X_img["train"].max())

In [ ]:
fig, ax = plt.subplots(2, 8, figsize=(14, 4))
for r, cls in enumerate([0, 1]):
    for a, i in zip(ax[r], np.where(y["train"] == cls)[0][:8]):
        a.imshow(X_img["train"][i], cmap="gray"); a.axis("off")
    ax[r][0].set_title(CLASS_NAMES[cls], loc="left")
plt.suptitle("PneumoniaMNIST training samples (original 28×28)"); plt.tight_layout()
plt.savefig(OUT / "samples.png", dpi=120); plt.show()

## 3. Preprocessing
* Images are already single-channel grayscale chest X-rays (uint8).
* Every image is resized **28×28 → 128×128** with bicubic interpolation so that the HOG configuration of Table 1 (8×8-pixel cells, 2×2-cell blocks) gives a 16×16 cell grid.
* Feature standardisation (`StandardScaler`) is fitted on the **training** split only, inside the SVM pipeline.

In [ ]:
def preprocess(img, size=IMG_SIZE):
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_CUBIC)

X_pre = {s: np.array([preprocess(im) for im in X_img[s]]) for s in SPLITS}
print({s: X_pre[s].shape for s in SPLITS})

## 4. HOG feature extraction

In [ ]:
def extract_hog(imgs):
    # HOG per image is independent -> spread over all CPU cores
    return np.array(Parallel(n_jobs=-1)(delayed(hog)(im, **HOG_PARAMS) for im in imgs), dtype=np.float32)

t0 = time.perf_counter()
F = {s: extract_hog(X_pre[s]) for s in SPLITS}
T_HOG = time.perf_counter() - t0
N_FEAT = F["train"].shape[1]
n_cells = IMG_SIZE // 8; n_blocks = n_cells - 1
print(f"HOG features per image: {N_FEAT}  (= {n_blocks}x{n_blocks} blocks x 2x2 cells x 9 orientations)")
print(f"HOG extraction for all {sum(len(y[s]) for s in SPLITS)} images: {T_HOG:.1f}s")

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(12, 6))
for c, i in enumerate([np.where(y["train"] == 0)[0][0], np.where(y["train"] == 0)[0][1],
                       np.where(y["train"] == 1)[0][0], np.where(y["train"] == 1)[0][1]]):
    _, vis = hog(X_pre["train"][i], visualize=True, **HOG_PARAMS)
    ax[0][c].imshow(X_pre["train"][i], cmap="gray"); ax[0][c].set_title(CLASS_NAMES[y["train"][i]]); ax[0][c].axis("off")
    ax[1][c].imshow(exposure.rescale_intensity(vis, in_range=(0, vis.max() + 1e-9)), cmap="gray")
    ax[1][c].set_title("HOG"); ax[1][c].axis("off")
plt.tight_layout(); plt.savefig(OUT / "hog_visualisation.png", dpi=120); plt.show()

## 5. Baseline: HOG + SVM

**Fast (GPU) validation scorer.** The GA needs thousands of SVM fits, so validation scores are computed with an exact shortcut: the features are standardised once (scaler fitted on train) and moved to the GPU; for any feature subset the squared distances ‖a−b‖² = ‖a‖² + ‖b‖² − 2a·b are built with one GPU matrix product, turned into the RBF kernel, and libsvm is trained on that pre-computed kernel (CPU threads, in parallel). Because standardisation is per-feature, the selected columns of the standardised matrix are exactly what `StandardScaler` would produce on the subset, so this is **the same model** as `SVC(kernel="rbf")` – just much faster (the kernel matrix product was ~90 % of the cost). The same scorer is used for the baseline tuning, the GA fitness and the GA-HOG tuning, so all model selection is done identically. (Equivalence is checked below.)

In [ ]:
scaler_all = StandardScaler().fit(F["train"])
Zt = {s: torch.from_numpy(scaler_all.transform(F[s]).astype(np.float32)).to(DEVICE)
      for s in ("train", "val")}                    # test is not needed for selection

def sq_dists(mask):
    # squared Euclidean distances (train-train, val-train) on the selected standardised features, on DEVICE
    idx = torch.from_numpy(np.flatnonzero(mask)).to(DEVICE)
    A, B = Zt["train"][:, idx], Zt["val"][:, idx]
    an, bn = (A * A).sum(1), (B * B).sum(1)
    return (an[:, None] + an[None] - 2 * A @ A.T).clamp_(min=0), (bn[:, None] + an[None] - 2 * B @ A.T).clamp_(min=0)

def svm_val_f1(D, n, C, g):
    # RBF-SVM (gamma = g / n) trained on the pre-computed train kernel, macro-F1 on val
    Dtr, Dva = D
    gamma = g / n
    Ktr = torch.exp(-gamma * Dtr).double().cpu().numpy()
    Kva = torch.exp(-gamma * Dva).double().cpu().numpy()
    svm = SVC(C=C, kernel="precomputed", class_weight="balanced").fit(Ktr, y["train"])
    return f1_score(y["val"], svm.predict(Kva), average="macro")

def val_macro_f1(mask, C, g):
    n = int(mask.sum())
    return svm_val_f1(sq_dists(mask), n, C, g) if n > 0 else 0.0

def tune_svm(mask):
    D, n = sq_dists(mask), int(mask.sum())            # distances computed once, reused for every (C, g)
    grid = [(C, g) for C in C_GRID for g in G_GRID]
    f1s = list(POOL.map(lambda cg: svm_val_f1(D, n, *cg), grid))
    rows = [dict(C=C, g=g, gamma=g / n, val_macro_f1=f) for (C, g), f in zip(grid, f1s)]
    # stable sort keeps grid order on ties -> the smaller C wins (simpler model)
    return pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False, kind="stable").reset_index(drop=True)

def make_svm(C, gamma):
    return make_pipeline(StandardScaler(), SVC(kernel="rbf", C=C, gamma=gamma, class_weight="balanced", cache_size=2000))

In [ ]:
FULL_MASK = np.ones(N_FEAT, bool)
t0 = time.perf_counter()
base_grid = tune_svm(FULL_MASK)
T_TUNE_BASE = time.perf_counter() - t0
BASE_C, BASE_G = base_grid.loc[0, "C"], base_grid.loc[0, "g"]
BASE_GAMMA = BASE_G / N_FEAT
print(f"Baseline tuning ({len(base_grid)} configs): {T_TUNE_BASE:.1f}s  ->  C={BASE_C}, gamma={BASE_G}/{N_FEAT}, "
      f"val macro-F1={base_grid.loc[0, 'val_macro_f1']:.4f}")
display(base_grid.pivot(index="C", columns="g", values="val_macro_f1").style.format("{:.4f}").background_gradient(cmap="Blues"))
base_grid.to_csv(OUT / "baseline_svm_tuning.csv", index=False)

In [ ]:
# Final baseline model: standard sklearn pipeline on the raw 8100-D HOG features (this is the timed model)
t0 = time.perf_counter()
base_model = make_svm(BASE_C, BASE_GAMMA).fit(F["train"], y["train"])
T_TRAIN_BASE = time.perf_counter() - t0
base_val_f1 = f1_score(y["val"], base_model.predict(F["val"]), average="macro")
print(f"Baseline training time: {T_TRAIN_BASE:.1f}s | support vectors: {base_model[-1].n_support_.sum()}")
print(f"Val macro-F1  sklearn pipeline = {base_val_f1:.4f}   fast scorer = {base_grid.loc[0, 'val_macro_f1']:.4f}  (should match)")

## 6. Genetic Algorithm for HOG feature selection

Each **chromosome** is a binary mask of length 8100 (1 = keep that HOG feature).

| GA component | Implementation |
|---|---|
| Population initialisation | each gene ~ Bernoulli(0.5) (random masks keeping ≈ half of the features) |
| Fitness evaluation | `fitness = val macro-F1 − λ · (n_selected / 8100)`, λ = 0.02, RBF-SVM with the baseline's tuned C and g (γ = g / n_selected) |
| Selection | tournament selection, k = 3 |
| Crossover | uniform crossover, p_c = 0.9 |
| Mutation | independent bit-flip of every gene with probability p_m |
| Elitism | the 2 best chromosomes are copied unchanged to the next generation |
| Termination | fixed number of generations (30) |

The penalty term means that, of two masks with the same F1, the smaller one wins; removing 10 % of the features is worth 0.002 of F1. Fitness values are cached per chromosome so unchanged elites are not re-evaluated, and all new chromosomes of a generation are evaluated in parallel (GPU kernels + threaded libsvm).

In [ ]:
LAMBDA = 0.02
N_GEN = 30

def run_ga(pop_size, p_mut, n_gen=N_GEN, p_cross=0.9, tourn_k=3, n_elite=2, p_init=0.5, lam=LAMBDA,
           C=BASE_C, g=BASE_G, seed=SEED, verbose=True):
    rng = np.random.default_rng(seed)
    cache = {}

    def evaluate(pop):
        keys = [np.packbits(ind).tobytes() for ind in pop]
        todo = {k: ind for k, ind in zip(keys, pop) if k not in cache}
        for k, f1 in zip(todo, POOL.map(lambda ind: val_macro_f1(ind, C, g), todo.values())):
            cache[k] = (f1 - lam * todo[k].sum() / N_FEAT, f1)
        return np.array([cache[k] for k in keys])

    def tournament(fit):
        cand = rng.choice(len(fit), tourn_k, replace=False)
        return cand[np.argmax(fit[cand])]

    t0 = time.perf_counter()
    pop = rng.random((pop_size, N_FEAT)) < p_init                         # 1) population initialisation
    history = []
    for gen in range(n_gen + 1):
        res = evaluate(pop)                                                # 2) fitness evaluation
        fit, f1 = res[:, 0], res[:, 1]
        b, n_sel = fit.argmax(), pop.sum(1)
        history.append(dict(generation=gen, best_fitness=fit[b], mean_fitness=fit.mean(), worst_fitness=fit.min(),
                            best_val_f1=f1[b], best_n_features=int(n_sel[b]), mean_n_features=n_sel.mean(),
                            unique_evaluations=len(cache), elapsed_s=time.perf_counter() - t0))
        if verbose and (gen % 5 == 0 or gen == n_gen):
            print(f"  gen {gen:2d} | best fit {fit[b]:.4f} (F1 {f1[b]:.4f}, {n_sel[b]} feats) | "
                  f"mean fit {fit.mean():.4f} | {time.perf_counter() - t0:.0f}s")
        if gen == n_gen:
            break
        children = [pop[i].copy() for i in np.argsort(-fit)[:n_elite]]    # elitism
        while len(children) < pop_size:
            p1, p2 = pop[tournament(fit)], pop[tournament(fit)]          # 3) tournament selection
            if rng.random() < p_cross:                                     # 4) uniform crossover
                swap = rng.random(N_FEAT) < 0.5
                c1, c2 = np.where(swap, p2, p1), np.where(swap, p1, p2)
            else:
                c1, c2 = p1.copy(), p2.copy()
            for c in (c1, c2):                                             # 5) bit-flip mutation
                c ^= rng.random(N_FEAT) < p_mut
                children.append(c)
        pop = np.array(children[:pop_size])
    return dict(best_mask=pop[b].copy(), best_fitness=fit[b], best_val_f1=f1[b], history=pd.DataFrame(history),
                time_s=time.perf_counter() - t0, evaluations=len(cache))

## 7. GA experiments (three GA settings)

Two GA parameters are varied (lab requirement: at least two settings):
* **Mutation probability** p_m: 0.005 (≈ 40 bit flips per child) vs 0.02 (≈ 160 flips)
* **Population size**: 20 vs 40

All other settings are the same (30 generations, tournament k=3, uniform crossover p_c=0.9, 2 elites, λ = 0.02, same seed). The setting with the highest **validation fitness** is selected; the test set is not used here.

In [ ]:
GA_SETTINGS = {
    "GA-1 (pop 20, pm 0.005)": dict(pop_size=20, p_mut=0.005),
    "GA-2 (pop 20, pm 0.02)":  dict(pop_size=20, p_mut=0.02),
    "GA-3 (pop 40, pm 0.005)": dict(pop_size=40, p_mut=0.005),
}
ga_runs = {}
for name, cfg in GA_SETTINGS.items():
    print(name)
    ga_runs[name] = run_ga(**cfg)
    ga_runs[name]["history"].to_csv(OUT / f"ga_history_{name.split()[0]}.csv", index=False)

In [ ]:
ga_summary = pd.DataFrame([{
    "GA setting": n, "Population": GA_SETTINGS[n]["pop_size"], "Mutation prob.": GA_SETTINGS[n]["p_mut"],
    "Generations": N_GEN, "SVM fits": r["evaluations"], "GA time (s)": r["time_s"],
    "Selected features": int(r["best_mask"].sum()), "Reduction (%)": 100 * (1 - r["best_mask"].mean()),
    "Best val macro-F1": r["best_val_f1"], "Best val fitness": r["best_fitness"]} for n, r in ga_runs.items()])
display(ga_summary)
ga_summary.to_csv(OUT / "ga_settings_comparison.csv", index=False)

BEST_GA = ga_summary.loc[ga_summary["Best val fitness"].idxmax(), "GA setting"]
GA_MASK = ga_runs[BEST_GA]["best_mask"]
T_GA = ga_runs[BEST_GA]["time_s"]
print(f"Selected GA setting (highest validation fitness): {BEST_GA}  ->  {GA_MASK.sum()} / {N_FEAT} features")
np.save(OUT / "ga_selected_mask.npy", GA_MASK)

## 8. GA-HOG + SVM

The SVM is re-tuned on the validation set using only the GA-selected features (same grid as the baseline), then a standard sklearn pipeline is trained on the selected training features.

As a **control**, five *random* masks with the same number of features are also scored on validation: if the GA mask is no better than a random subset of the same size, the GA has not learned anything beyond "fewer features".

In [ ]:
t0 = time.perf_counter()
ga_grid = tune_svm(GA_MASK)
T_TUNE_GA = time.perf_counter() - t0
GA_C, GA_G = ga_grid.loc[0, "C"], ga_grid.loc[0, "g"]
GA_GAMMA = GA_G / GA_MASK.sum()
print(f"GA-HOG tuning: {T_TUNE_GA:.1f}s -> C={GA_C}, gamma={GA_G}/{GA_MASK.sum()}, val macro-F1={ga_grid.loc[0, 'val_macro_f1']:.4f}")
display(ga_grid.pivot(index="C", columns="g", values="val_macro_f1").style.format("{:.4f}").background_gradient(cmap="Greens"))
ga_grid.to_csv(OUT / "ga_hog_svm_tuning.csv", index=False)

t0 = time.perf_counter()
ga_model = make_svm(GA_C, GA_GAMMA).fit(F["train"][:, GA_MASK], y["train"])
T_TRAIN_GA = time.perf_counter() - t0
ga_val_f1 = f1_score(y["val"], ga_model.predict(F["val"][:, GA_MASK]), average="macro")
print(f"GA-HOG training time: {T_TRAIN_GA:.1f}s | support vectors: {ga_model[-1].n_support_.sum()} | val macro-F1 {ga_val_f1:.4f}")

In [ ]:
rng = np.random.default_rng(SEED)
k = int(GA_MASK.sum())
rand_masks = []
for _ in range(5):
    m = np.zeros(N_FEAT, bool); m[rng.choice(N_FEAT, k, replace=False)] = True
    rand_masks.append(m)
rand_f1 = list(POOL.map(lambda m: val_macro_f1(m, GA_C, GA_G), rand_masks))
val_cmp = pd.DataFrame([
    {"Model (validation set)": "HOG + SVM (all features)", "Features": N_FEAT, "Val macro-F1": base_val_f1},
    {"Model (validation set)": f"Random subset (5 draws, mean ± sd)", "Features": k,
     "Val macro-F1": f"{np.mean(rand_f1):.4f} ± {np.std(rand_f1):.4f}"},
    {"Model (validation set)": f"GA-HOG + SVM ({BEST_GA})", "Features": k, "Val macro-F1": ga_val_f1}])
display(val_cmp)
val_cmp.to_csv(OUT / "validation_comparison.csv", index=False)

## 9. Final evaluation on the test set (run once, after model selection)

All choices – SVM hyper-parameters, GA setting, GA mask – were made on the validation set above. The test set is used here for the first and only time.

**Specificity (multiclass averaging):** for each class *c* we compute the one-vs-rest specificity TN_c / (TN_c + FP_c) and report the unweighted (macro) mean over the classes. For a binary problem the specificity of one class equals the recall of the other, so macro-specificity equals macro-recall and balanced accuracy.

In [ ]:
def all_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    spec = [(cm.sum() - cm[c].sum() - cm[:, c].sum() + cm[c, c]) / (cm.sum() - cm[c].sum()) for c in range(len(cm))]
    return {
        "Accuracy (%)": 100 * accuracy_score(y_true, y_pred),
        "Precision (Macro, %)": 100 * precision_score(y_true, y_pred, average="macro"),
        "Recall (Macro, %)": 100 * recall_score(y_true, y_pred, average="macro"),
        "Macro-F1 (%)": 100 * f1_score(y_true, y_pred, average="macro"),
        "Weighted-F1 (%)": 100 * f1_score(y_true, y_pred, average="weighted"),
        "Balanced accuracy (%)": 100 * balanced_accuracy_score(y_true, y_pred),
        "Specificity (Macro, %)": 100 * np.mean(spec),
        "Cohen's kappa": cohen_kappa_score(y_true, y_pred),
        "Matthews correlation coefficient": matthews_corrcoef(y_true, y_pred),
    }

def timed_predict(model, X, reps=5):
    best = np.inf
    for _ in range(reps):
        t0 = time.perf_counter(); p = model.predict(X); best = min(best, time.perf_counter() - t0)
    return p, best

pred_base, T_INF_BASE = timed_predict(base_model, F["test"])
pred_ga, T_INF_GA = timed_predict(ga_model, F["test"][:, GA_MASK])
m_base, m_ga = all_metrics(y["test"], pred_base), all_metrics(y["test"], pred_ga)

In [ ]:
table1 = pd.DataFrame({
    "Parameter": ["Medical dataset", "Image resolution", "HOG orientations", "Pixels per cell", "Cells per block",
                  "Block normalization", "HOG feature dimension", "Feature selection", "Classifier", "SVM kernel",
                  "Class weighting", "Hyperparameter tuning", "Selected C / gamma", "Data split", "Random seed"],
    "Baseline: HOG + SVM": ["PneumoniaMNIST (Kaggle: rijulshr/pneumoniamnist)", "128 × 128", 9, "8 × 8", "2 × 2", "L2-Hys",
                            N_FEAT, "None", "SVM", "RBF", "balanced", "Validation set (macro-F1)",
                            f"C={BASE_C}, γ={BASE_G}/{N_FEAT}", "Identical fixed split (3882 / 524 / 624)", SEED],
    "Proposed: GA-HOG + SVM": ["PneumoniaMNIST (Kaggle: rijulshr/pneumoniamnist)", "128 × 128", 9, "8 × 8", "2 × 2", "L2-Hys",
                               int(GA_MASK.sum()), f"Genetic Algorithm – {BEST_GA}", "SVM", "RBF", "balanced",
                               "Validation set (macro-F1)", f"C={GA_C}, γ={GA_G}/{GA_MASK.sum()}",
                               "Identical fixed split (3882 / 524 / 624)", SEED]})
print("Table 1. Experimental configuration and HOG feature-extraction parameters")
display(table1.style.hide(axis="index"))
table1.to_csv(OUT / "table1_configuration.csv", index=False)

table2 = pd.DataFrame({"Metric": list(m_base), "HOG + SVM (Baseline)": list(m_base.values()),
                       "GA-HOG + SVM (Proposed)": list(m_ga.values())})
table2["Improvement"] = table2["GA-HOG + SVM (Proposed)"] - table2["HOG + SVM (Baseline)"]
print("Table 2. Main performance comparison on the TEST set: Without GA versus With GA  "
      "(Improvement = Proposed − Baseline; percentage points for % metrics)")
display(table2.style.hide(axis="index").format({c: "{:+.2f}" if c == "Improvement" else "{:.2f}" for c in table2.columns[1:]})
        .format("{:.4f}", subset=pd.IndexSlice[7:8, table2.columns[1:3]])
        .format("{:+.4f}", subset=pd.IndexSlice[7:8, ["Improvement"]]))
table2.to_csv(OUT / "table2_performance_test.csv", index=False)

for name, p in (("HOG + SVM (Baseline)", pred_base), ("GA-HOG + SVM (Proposed)", pred_ga)):
    print("=" * 64, "\n", name, "\n", classification_report(y["test"], p, target_names=CLASS_NAMES, digits=4))

In [ ]:
n_test = len(y["test"])
table3 = pd.DataFrame({
    "Cost item": ["HOG feature dimension", "HOG extraction, all 5030 images (s)", "GA feature selection search (s)",
                  "SVM hyper-parameter tuning on validation (s)", "Final SVM training (s)",
                  f"Test inference, {n_test} images (s)", "Inference per image (ms)", "Support vectors",
                  "Feature memory per image (KB, float32)", "Total offline time (s)"],
    "HOG + SVM (Baseline)": [N_FEAT, T_HOG, 0.0, T_TUNE_BASE, T_TRAIN_BASE, T_INF_BASE, 1000 * T_INF_BASE / n_test,
                             base_model[-1].n_support_.sum(), N_FEAT * 4 / 1024, T_HOG + T_TUNE_BASE + T_TRAIN_BASE],
    "GA-HOG + SVM (Proposed)": [int(GA_MASK.sum()), T_HOG, T_GA, T_TUNE_GA, T_TRAIN_GA, T_INF_GA, 1000 * T_INF_GA / n_test,
                                ga_model[-1].n_support_.sum(), GA_MASK.sum() * 4 / 1024,
                                T_HOG + T_GA + T_TUNE_BASE + T_TUNE_GA + T_TRAIN_GA]})
table3["Change (%)"] = 100 * (table3.iloc[:, 2].astype(float) / table3.iloc[:, 1].replace(0, np.nan).astype(float) - 1)
print("Table 3. Computational cost (GA-HOG total includes the baseline tuning, because the GA uses the baseline's C and g)")
display(table3.style.hide(axis="index").format(precision=2, na_rep="—"))
table3.to_csv(OUT / "table3_computational_cost.csv", index=False)

## 10. Plots

In [ ]:
# GA fitness across generations (best / mean) for every GA setting
fig, ax = plt.subplots(1, len(ga_runs), figsize=(5.5 * len(ga_runs), 4), sharey=True)
for a, (name, r) in zip(ax, ga_runs.items()):
    h = r["history"]
    a.plot(h.generation, h.best_fitness, "o-", ms=3, label="best fitness")
    a.plot(h.generation, h.mean_fitness, "s--", ms=3, label="mean fitness")
    a.fill_between(h.generation, h.worst_fitness, h.best_fitness, alpha=0.12)
    a.axhline(base_val_f1 - LAMBDA, color="gray", ls=":", label="baseline (all feats) fitness")
    a.set_title(name); a.set_xlabel("Generation"); a.grid(alpha=0.3)
ax[0].set_ylabel("Fitness = val macro-F1 − λ·(n/8100)"); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "ga_fitness_per_setting.png", dpi=130); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for name, r in ga_runs.items():
    h = r["history"]
    ax[0].plot(h.generation, h.best_fitness, "o-", ms=3, label=name)
    ax[1].plot(h.generation, h.best_val_f1, "o-", ms=3, label=name)
    ax[2].plot(h.generation, h.best_n_features, "o-", ms=3, label=name)
ax[1].axhline(base_val_f1, color="k", ls=":", label="baseline val macro-F1")
for a, t in zip(ax, ["Best fitness", "Val macro-F1 of best chromosome", "Features in best chromosome"]):
    a.set_title(t); a.set_xlabel("Generation"); a.grid(alpha=0.3); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "ga_settings_comparison.png", dpi=130); plt.show()

In [ ]:
# Number of features before and after selection
names = ["HOG (no selection)"] + list(ga_runs)
counts = [N_FEAT] + [int(r["best_mask"].sum()) for r in ga_runs.values()]
colors = ["#7f7f7f"] + ["#2ca02c" if n == BEST_GA else "#98df8a" for n in ga_runs]
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(names, counts, color=colors)
for b_, c in zip(bars, counts):
    ax.text(b_.get_x() + b_.get_width() / 2, c + 80, f"{c}\n({100 * c / N_FEAT:.1f}%)", ha="center", fontsize=9)
ax.set_ylabel("Number of HOG features"); ax.set_ylim(0, N_FEAT * 1.18)
ax.set_title("HOG feature count before vs after GA selection (dark green = selected GA setting)")
plt.xticks(fontsize=8); plt.tight_layout(); plt.savefig(OUT / "feature_count_before_after.png", dpi=130); plt.show()

In [ ]:
# Confusion matrices + metric comparison on the test set
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5), gridspec_kw=dict(width_ratios=[1, 1, 2.2]))
for a, (n, p) in zip(ax, (("HOG + SVM (Baseline)", pred_base), ("GA-HOG + SVM (Proposed)", pred_ga))):
    ConfusionMatrixDisplay(confusion_matrix(y["test"], p), display_labels=CLASS_NAMES).plot(ax=a, cmap="Blues", colorbar=False)
    a.set_title(n)
pct = table2.iloc[:7]
x = np.arange(len(pct)); w = 0.38
ax[2].bar(x - w / 2, pct["HOG + SVM (Baseline)"], w, label="HOG + SVM", color="#7f7f7f")
ax[2].bar(x + w / 2, pct["GA-HOG + SVM (Proposed)"], w, label="GA-HOG + SVM", color="#2ca02c")
ax[2].set_xticks(x); ax[2].set_xticklabels([m.replace(" (", "\n(") for m in pct.Metric], fontsize=7)
lo = min(pct.iloc[:, 1:3].values.min() - 5, 70); ax[2].set_ylim(lo, 100); ax[2].legend(); ax[2].grid(axis="y", alpha=0.3)
ax[2].set_title("Test-set metrics (%)")
plt.tight_layout(); plt.savefig(OUT / "test_confusion_and_metrics.png", dpi=130); plt.show()

In [ ]:
# Where are the selected features? HOG vector layout = (block_row, block_col, cell_row, cell_col, orientation)
sel = GA_MASK.reshape(n_blocks, n_blocks, 2, 2, 9)
cell_count = np.zeros((n_cells, n_cells)); cell_total = np.zeros((n_cells, n_cells))
for i in range(2):
    for j in range(2):
        cell_count[i:i + n_blocks, j:j + n_blocks] += sel[:, :, i, j, :].sum(-1)
        cell_total[i:i + n_blocks, j:j + n_blocks] += 9
frac_cell = cell_count / cell_total
frac_ori = sel.reshape(-1, 9).mean(0)

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
ax[0].imshow(X_pre["train"].mean(0), cmap="gray"); ax[0].set_title("Mean training image"); ax[0].axis("off")
im = ax[1].imshow(frac_cell, cmap="viridis", extent=(0, IMG_SIZE, IMG_SIZE, 0))
ax[1].imshow(X_pre["train"].mean(0), cmap="gray", alpha=0.35)
ax[1].set_title("Fraction of HOG features kept per 8×8 cell"); ax[1].axis("off"); plt.colorbar(im, ax=ax[1], fraction=0.046)
ax[2].bar(np.arange(9) * 20 + 10, 100 * frac_ori, width=16, color="#2ca02c")
ax[2].axhline(100 * GA_MASK.mean(), color="k", ls=":", label="overall keep rate")
ax[2].set_xlabel("Orientation bin centre (degrees)"); ax[2].set_ylabel("% kept"); ax[2].legend()
ax[2].set_title("Kept features per orientation bin"); ax[2].set_ylim(0, 100)
plt.tight_layout(); plt.savefig(OUT / "selected_feature_map.png", dpi=130); plt.show()
print(f"Cell keep-rate range: {frac_cell.min():.2f} – {frac_cell.max():.2f}; orientation keep-rate range: "
      f"{frac_ori.min():.2f} – {frac_ori.max():.2f}")

In [ ]:
# Computational time comparison
items = ["HOG extraction", "GA search", "SVM tuning", "Final training", "Test inference"]
tb = [T_HOG, 0, T_TUNE_BASE, T_TRAIN_BASE, T_INF_BASE]
tg = [T_HOG, T_GA, T_TUNE_BASE + T_TUNE_GA, T_TRAIN_GA, T_INF_GA]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(items)); w = 0.38
ax[0].bar(x - w / 2, tb, w, label="HOG + SVM", color="#7f7f7f"); ax[0].bar(x + w / 2, tg, w, label="GA-HOG + SVM", color="#2ca02c")
ax[0].set_xticks(x); ax[0].set_xticklabels(items); ax[0].set_ylabel("seconds (log)"); ax[0].set_yscale("symlog", linthresh=1)
ax[0].legend(); ax[0].set_title("Computational time per stage"); ax[0].grid(axis="y", alpha=0.3)
ax[1].bar(list(ga_runs), [r["time_s"] / 60 for r in ga_runs.values()], color="#98df8a")
ax[1].set_ylabel("minutes"); ax[1].set_title("GA search time per setting"); ax[1].tick_params(axis="x", labelsize=8)
plt.tight_layout(); plt.savefig(OUT / "computational_time.png", dpi=130); plt.show()

## 11. Report

The report below is generated from the numbers of **this** run (so it always matches the tables and plots above) and is also saved to `results/report.md`.

In [ ]:
d = table2.set_index("Metric")
B, G, I = d["HOG + SVM (Baseline)"], d["GA-HOG + SVM (Proposed)"], d["Improvement"]
k_sel = int(GA_MASK.sum()); red = 100 * (1 - k_sel / N_FEAT)
dF1, dBA, dMCC = I["Macro-F1 (%)"], I["Balanced accuracy (%)"], I["Matthews correlation coefficient"]
val_gain = 100 * (ga_val_f1 - base_val_f1)
rand_gap = 100 * (ga_val_f1 - np.mean(rand_f1))
tc = table3.set_index("Cost item")
t_base_total, t_ga_total = tc.loc["Total offline time (s)"].iloc[:2]
inf_b, inf_g = tc.loc["Inference per image (ms)"].iloc[:2]
cm_b, cm_g = confusion_matrix(y["test"], pred_base), confusion_matrix(y["test"], pred_ga)
recall_n = (100 * cm_b[0, 0] / cm_b[0].sum(), 100 * cm_g[0, 0] / cm_g[0].sum())
recall_p = (100 * cm_b[1, 1] / cm_b[1].sum(), 100 * cm_g[1, 1] / cm_g[1].sum())

if dF1 >= 0.5:
    verdict = (f"**Yes.** GA-based feature selection improved the test macro-F1 by {dF1:+.2f} percentage points "
               f"(balanced accuracy {dBA:+.2f} pp, MCC {dMCC:+.4f}) while removing {red:.1f}% of the HOG features.")
elif dF1 > -0.5:
    verdict = (f"**Partly.** The GA removed {red:.1f}% of the HOG features ({N_FEAT} → {k_sel}) with essentially unchanged "
               f"test performance (macro-F1 {dF1:+.2f} pp, balanced accuracy {dBA:+.2f} pp, MCC {dMCC:+.4f}). "
               f"The main benefit is a smaller, cheaper model rather than a clear accuracy gain.")
else:
    verdict = (f"**No.** Although the GA removed {red:.1f}% of the features and improved the *validation* macro-F1 by "
               f"{val_gain:+.2f} pp, the test macro-F1 changed by {dF1:+.2f} pp (balanced accuracy {dBA:+.2f} pp), "
               f"i.e. the validation gain did not generalise.")

settings_lines = "\n".join(
    f"| {r['GA setting']} | {r['SVM fits']} | {r['GA time (s)']:.0f} | {r['Selected features']} | "
    f"{r['Reduction (%)']:.1f} | {100 * r['Best val macro-F1']:.2f} | {r['Best val fitness']:.4f} |"
    for _, r in ga_summary.iterrows())
metric_lines = "\n".join(
    f"| {m} | {B[m]:.2f} | {G[m]:.2f} | {I[m]:+.2f} |" if "%" in m else f"| {m} | {B[m]:.4f} | {G[m]:.4f} | {I[m]:+.4f} |"
    for m in d.index)
device_txt = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU only"

report = f'''
### Lab 07 report: GA-based HOG feature selection for pneumonia classification

**Research question.** Can Genetic Algorithm-based feature selection improve the classification performance of HOG features while reducing feature dimensionality on medical images?

**Answer.** {verdict}

#### Dataset reference
* **PneumoniaMNIST** (MedMNIST v2): J. Yang, R. Shi, D. Wei, *et al.*, "MedMNIST v2 – A large-scale lightweight benchmark for 2D and 3D biomedical image classification", *Scientific Data* 10, 41 (2023).
* Original images: D. S. Kermany, M. Goldbaum, *et al.*, "Identifying Medical Diagnoses and Treatable Diseases by Image-Based Deep Learning", *Cell* 172(5), 1122–1131 (2018).
* Copy used here: Kaggle `rijulshr/pneumoniamnist` (`pneumoniamnist.npz`, 28×28 grayscale chest X-rays, 2 classes: Normal / Pneumonia), downloaded with `kagglehub`.
* Split used (fixed, identical for all models): train {len(y["train"])} ({(y["train"] == 0).sum()} Normal / {(y["train"] == 1).sum()} Pneumonia), validation {len(y["val"])}, test {len(y["test"])} ({(y["test"] == 0).sum()} Normal / {(y["test"] == 1).sum()} Pneumonia). Note: this Kaggle copy has fewer Normal training images than the official MedMNIST v2 training split, so training is strongly imbalanced (≈1:{(y['train'] == 1).sum() / (y['train'] == 0).sum():.0f}); `class_weight="balanced"` is used in every SVM. An exact-duplicate check found {DUPS['train∩val']} images shared by train and validation, {DUPS['train∩test']} by train and test and {DUPS['val∩test']} by validation and test; the official split was kept unchanged.

#### Method (summary)
Images were up-sampled to 128×128 and described with HOG (9 orientations, 8×8 cells, 2×2 blocks, L2-Hys → {N_FEAT} features). The baseline trains an RBF-SVM on all features; C and γ were tuned on the validation set ({len(C_GRID) * len(G_GRID)}-point grid, γ = g / n_features, macro-F1). The proposed method searches a binary feature mask with a GA (random Bernoulli(0.5) initialisation, tournament selection k=3, uniform crossover p_c=0.9, bit-flip mutation, 2 elites, 30 generations) whose fitness is *validation macro-F1 − {LAMBDA}·(selected/total)*. Three GA settings were compared and the one with the best validation fitness (**{BEST_GA}**) was kept; the SVM was then re-tuned on the selected features. The test set was used once, after all of these choices.

#### GA settings (validation set)
| GA setting | SVM fits | GA time (s) | Selected features | Reduction (%) | Best val macro-F1 (%) | Best val fitness |
|---|---|---|---|---|---|---|
{settings_lines}

On the validation set the selected GA mask scored {100 * ga_val_f1:.2f}% macro-F1 vs {100 * base_val_f1:.2f}% for all features ({val_gain:+.2f} pp) and vs {100 * np.mean(rand_f1):.2f} ± {100 * np.std(rand_f1):.2f}% for random subsets of the same size ({rand_gap:+.2f} pp), so the GA found a better-than-random subset on the data it optimised.

#### Test-set results (Table 2)
| Metric | HOG + SVM | GA-HOG + SVM | Improvement |
|---|---|---|---|
{metric_lines}

Specificity is the macro average of the one-vs-rest specificities; for two classes it equals macro recall / balanced accuracy. Normal-class recall: {recall_n[0]:.1f}% → {recall_n[1]:.1f}%; Pneumonia recall: {recall_p[0]:.1f}% → {recall_p[1]:.1f}%.

#### Discussion
* **Validation vs test.** Both models score much lower on the test set than on validation (baseline macro-F1 {100 * base_val_f1:.1f}% on validation vs {B["Macro-F1 (%)"]:.1f}% on test). The errors are mostly Normal X-rays predicted as Pneumonia (Normal recall ≈ {recall_n[0]:.0f}%): the training set is {100 * y['train'].mean():.0f}% Pneumonia while the test set is {100 * y['test'].mean():.1f}% Pneumonia, and the PneumoniaMNIST test images come from a slightly different distribution. Any GA gain measured on validation is therefore partly specific to the validation set: the GA optimises validation macro-F1 directly, so the validation score of the selected mask is optimistically biased, and the test set is the only unbiased check.
* **Strengths.** (1) The GA is model-agnostic and needs no gradients; it can optimise any metric (here macro-F1, which suits the imbalanced data) together with a size penalty. (2) It reduced the descriptor from {N_FEAT} to {k_sel} features ({red:.1f}% fewer), which reduces feature memory and SVM kernel cost (inference {inf_b:.2f} → {inf_g:.2f} ms per image). (3) The selected mask is interpretable: the selected-feature map shows which 8×8 cells and orientation bins the classifier relies on. (4) It consistently beat random subsets of the same size on validation.
* **Limitations.** (1) The search space has 2^{N_FEAT} masks; 30 generations with small populations explore only a tiny part of it, and most of the size reduction comes from the 50 % random initialisation, while the λ={LAMBDA} penalty only slowly removes further features. (2) Fitness uses a single validation set, so the GA can overfit it (selection bias); k-fold fitness or a separate selection set would be more reliable but several times more expensive. (3) The result depends on the GA hyper-parameters (population, mutation rate, λ, seed); only one seed was run per setting, so differences between settings of a few tenths of a percent are within noise. (4) HOG on 28×28 images up-sampled to 128×128 contains a lot of redundant information; the GA cannot add information that HOG does not capture.
* **Computational cost.** HOG extraction is the same for both methods ({T_HOG:.1f}s). The GA needed {ga_runs[BEST_GA]["evaluations"]} SVM trainings for the selected setting ({T_GA:.0f}s; {sum(r["time_s"] for r in ga_runs.values()):.0f}s for all three settings) on {device_txt}, versus {len(C_GRID) * len(G_GRID)} trainings for the baseline tuning ({T_TUNE_BASE:.1f}s). Total offline time: baseline {t_base_total:.0f}s vs GA-HOG {t_ga_total:.0f}s ({t_ga_total / t_base_total:.1f}×). This was only practical because each RBF kernel was built with a single matrix product on {device_txt} and libsvm ran on the pre-computed kernel in parallel threads; for comparison, one standard `sklearn` SVC fit on all features took {T_TRAIN_BASE:.1f}s, so the same GA with plain `SVC` would take hours. Once trained, the GA-HOG model is cheaper: final training {tc.loc["Final SVM training (s)"].iloc[0]:.1f}s → {tc.loc["Final SVM training (s)"].iloc[1]:.1f}s, and inference is faster because fewer features enter the kernel.

#### Conclusion
{verdict} GA feature selection is a cheap-at-inference, expensive-at-training way to compress HOG descriptors for medical images; to turn the validation gains into reliable test gains it should be combined with cross-validated fitness and repeated runs over several seeds.
'''
(OUT / "report.md").write_text(report, encoding="utf-8")
Markdown(report)

In [ ]:
# Bundle all tables, plots, GA histories, the GA mask and the report (downloaded automatically in Colab)
with zipfile.ZipFile("lab07_results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.iterdir()):
        zf.write(f)
print("Saved:", ", ".join(f.name for f in sorted(OUT.iterdir())))
try:
    from google.colab import files
    files.download("lab07_results.zip")
except ImportError:
    print("Not running in Colab – results are in ./results and lab07_results.zip")